# Day 105 — Handwritten Digit Classification using ANN (MNIST)

**Problem:** look at an image of a handwritten digit and predict which digit (0-9) it is. This is a **multi-class classification** problem on **image data**.

**Dataset:** MNIST: 70,000 grayscale images of size **28 × 28** (60,000 for training, 10,000 for testing), downloaded automatically by Keras.
If the download is not possible (no internet), the notebook falls back to scikit-learn's small 8 × 8 `digits` dataset so every cell still runs
(accuracy numbers will then differ).

**Requirements:** `pip install tensorflow numpy matplotlib scikit-learn`

**Workflow:**
1. Load and visualize the images
2. Preprocess: scale pixels to 0-1 and flatten each image into a vector
3. Build an ANN with a **softmax** output layer
4. Compile with **sparse categorical cross entropy** and train
5. Plot curves, evaluate, inspect the confusion matrix and mistakes
6. Predict a single image
7. Improve (dropout, early stopping, wider layers) and see why **scaling** matters
8. Look at what the first layer learned, then save the model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow version:", tf.__version__)

## 1. Load the data

In [ ]:
try:
    (X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
    DATA_NAME = "MNIST (28 x 28)"
except Exception as e:
    print("Could not download MNIST:", type(e).__name__, "-> falling back to scikit-learn's 8x8 digits dataset")
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    d = load_digits()
    imgs = (d.images / 16 * 255).astype("uint8")                  # same 0-255 pixel range as MNIST
    X_train, X_test, y_train, y_test = train_test_split(imgs, d.target, test_size=0.2, stratify=d.target, random_state=SEED)
    DATA_NAME = "scikit-learn digits (8 x 8)"

IMG_H, IMG_W = X_train.shape[1:]
N_CLASSES = len(np.unique(y_train))
print("Dataset         :", DATA_NAME)
print("X_train shape   :", X_train.shape, "| y_train shape:", y_train.shape)
print("X_test shape    :", X_test.shape, "| y_test shape :", y_test.shape)
print("Pixel dtype/range:", X_train.dtype, X_train.min(), "to", X_train.max())
print("Classes         :", N_CLASSES)

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for ax, i in zip(axes.ravel(), range(20)):
    ax.imshow(X_train[i], cmap="gray"); ax.set_title(f"label {y_train[i]}", fontsize=9); ax.axis("off")
plt.suptitle(f"First 20 training images ({DATA_NAME})"); plt.tight_layout(); plt.show()

counts = np.bincount(y_train)
plt.figure(figsize=(6, 3)); plt.bar(range(N_CLASSES), counts); plt.xticks(range(N_CLASSES))
plt.title("Images per digit in the training set"); plt.xlabel("digit"); plt.show()

## 2. Preprocessing

- **Scale** pixel values from 0-255 to 0-1 (neural networks train much better on small, consistent inputs)
- The ANN expects a **vector**, so each image is **flattened** (`28 × 28 = 784` numbers). We do this with a `Flatten` layer inside the model.
- Labels stay as integers 0-9 (we will use *sparse* categorical cross entropy, so no one-hot encoding is needed)

In [ ]:
X_train_s = (X_train / 255.0).astype("float32")
X_test_s  = (X_test / 255.0).astype("float32")

print("After scaling -> min:", X_train_s.min(), "max:", X_train_s.max())
print("Each image becomes a vector of length", IMG_H * IMG_W)

## 3. Build the ANN

`Flatten (784) → Dense(128, ReLU) → Dense(32, ReLU) → Dense(10, softmax)`

- Hidden layers: **ReLU**
- Output: **10 nodes with softmax**, so the outputs are class probabilities that sum to 1

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(IMG_H, IMG_W)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(N_CLASSES, activation="softmax"),
], name="mnist_ann")

model.summary()

n_in = IMG_H * IMG_W
sizes = [n_in, 128, 32, N_CLASSES]
print("Expected parameters:", sum(sizes[i] * sizes[i + 1] + sizes[i + 1] for i in range(len(sizes) - 1)))

## 4. Compile and train

- **Loss:** `sparse_categorical_crossentropy` (labels are integers). Use `categorical_crossentropy` only if labels are one-hot encoded.
- **Optimizer:** Adam
- **validation_split=0.2:** Keras holds out the last 20% of the training data to monitor generalization

In [ ]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = model.fit(X_train_s, y_train, epochs=25, batch_size=32, validation_split=0.2, verbose=0)

h = history.history
print(f"Final epoch: train loss = {h['loss'][-1]:.4f}, val loss = {h['val_loss'][-1]:.4f}")
print(f"             train acc  = {h['accuracy'][-1]:.4f}, val acc  = {h['val_accuracy'][-1]:.4f}")
print(f"Lowest validation loss {min(h['val_loss']):.4f} at epoch {int(np.argmin(h['val_loss'])) + 1}")

In [ ]:
def plot_history(history, title=""):
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="validation")
    axes[0].set_title(f"Loss {title}"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(h["accuracy"], label="train"); axes[1].plot(h["val_accuracy"], label="validation")
    axes[1].set_title(f"Accuracy {title}"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history(history, "(baseline ANN)")

**Reading the curves:** if the validation loss stops improving (or rises) while the training loss keeps falling, the network is starting to **overfit**: it memorizes the training images instead of learning general digit shapes.

## 5. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"Test loss = {test_loss:.4f} | Test accuracy = {test_acc:.4f}")

proba = model.predict(X_test_s, verbose=0)           # shape (n_test, 10): class probabilities
y_pred = proba.argmax(axis=1)                         # most probable class
print("Probabilities for one image:", proba[0].round(3), "| sum =", proba[0].sum().round(3))
print("Predicted digit:", y_pred[0], "| true digit:", y_test[0])

In [ ]:
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xlabel("predicted digit"); ax.set_ylabel("true digit"); ax.set_title("Confusion matrix")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

# the most common confusions (off-diagonal cells)
off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:5], off.shape))[0]
print("Most common mistakes (true -> predicted : count):")
for t, p in top:
    print(f"  {t} -> {p} : {off[t, p]}")

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print("Misclassified test images:", len(wrong), "of", len(y_test))

fig, axes = plt.subplots(2, 8, figsize=(15, 4.2))
for ax, i in zip(axes.ravel(), wrong[:16]):
    ax.imshow(X_test[i], cmap="gray")
    ax.set_title(f"true {y_test[i]}, pred {y_pred[i]}\n({proba[i].max():.0%} sure)", fontsize=8); ax.axis("off")
plt.suptitle("Some mistakes the model makes"); plt.tight_layout(); plt.show()

## 6. Predict a single image and look at the probabilities

In [ ]:
idx = 0
image = X_test[idx]
p = model.predict(image[np.newaxis] / 255.0, verbose=0)[0]       # add a batch dimension of 1

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].imshow(image, cmap="gray"); axes[0].set_title(f"true label = {y_test[idx]}"); axes[0].axis("off")
axes[1].bar(range(N_CLASSES), p); axes[1].set_xticks(range(N_CLASSES))
axes[1].set_title(f"Predicted digit = {p.argmax()} ({p.max():.1%})"); axes[1].set_xlabel("digit"); axes[1].set_ylabel("probability")
plt.tight_layout(); plt.show()

### Predicting your own handwritten digit (optional)

Draw a digit in white on a black background (or any dark-on-light image, which we invert), save it as `my_digit.png`, then run this cell.
The model expects the **same format as the training images**: grayscale, the same size, scaled to 0-1.

In [ ]:
import os
if os.path.exists("my_digit.png"):
    from PIL import Image, ImageOps
    img = Image.open("my_digit.png").convert("L").resize((IMG_W, IMG_H))
    arr = np.array(img).astype("float32")
    if arr.mean() > 127:                       # mostly light background -> invert so the digit is light on dark like MNIST
        arr = 255 - arr
    p_own = model.predict((arr / 255.0)[np.newaxis], verbose=0)[0]
    plt.imshow(arr, cmap="gray"); plt.axis("off"); plt.title(f"Predicted digit: {p_own.argmax()} ({p_own.max():.1%})"); plt.show()
else:
    print("my_digit.png not found, skipping this optional cell.")

## 7. Why scaling matters (experiment)

Same model, but trained on the **raw 0-255 pixels**.

In [ ]:
def build_baseline():
    m = keras.Sequential([
        keras.Input(shape=(IMG_H, IMG_W)), layers.Flatten(),
        layers.Dense(128, activation="relu"), layers.Dense(32, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

m_raw = build_baseline()
hist_raw = m_raw.fit(X_train.astype("float32"), y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
m_scaled = build_baseline()
hist_scaled = m_scaled.fit(X_train_s, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)

print("Validation accuracy after 10 epochs")
print("  raw pixels (0-255):", round(hist_raw.history["val_accuracy"][-1], 4))
print("  scaled pixels (0-1):", round(hist_scaled.history["val_accuracy"][-1], 4))

plt.figure(figsize=(6, 4))
plt.plot(hist_raw.history["val_accuracy"], label="raw 0-255"); plt.plot(hist_scaled.history["val_accuracy"], label="scaled 0-1")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.grid(alpha=0.3); plt.title("Effect of input scaling"); plt.show()

## 8. Improving the model

- **Dropout** to reduce overfitting
- **EarlyStopping** to stop at the best validation loss
- A **wider first layer** for more capacity

In [ ]:
def build_improved(units=(256, 128), dropout=0.2):
    layer_list = [keras.Input(shape=(IMG_H, IMG_W)), layers.Flatten()]
    for u in units:
        layer_list += [layers.Dense(u, activation="relu"), layers.Dropout(dropout)]
    layer_list.append(layers.Dense(N_CLASSES, activation="softmax"))
    m = keras.Sequential(layer_list, name="mnist_ann_improved")
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

model2 = build_improved()
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
history2 = model2.fit(X_train_s, y_train, epochs=50, batch_size=64, validation_split=0.2, callbacks=[early_stop], verbose=0)
print("Stopped after", len(history2.history["loss"]), "epochs")
plot_history(history2, "(improved ANN)")

_, acc2 = model2.evaluate(X_test_s, y_test, verbose=0)
print(f"Test accuracy: baseline = {test_acc:.4f} | improved = {acc2:.4f}")

## 9. What did the first layer learn?

Each of the 128 first-layer neurons has one weight per pixel. Reshaping those weights back to the image shape shows the pattern each neuron responds to.

In [ ]:
W1 = model.layers[1].get_weights()[0]            # layers[0] is Flatten, layers[1] is the first Dense layer
print("First Dense layer weights:", W1.shape, "(pixels x neurons)")

fig, axes = plt.subplots(4, 8, figsize=(13, 7))
for ax, k in zip(axes.ravel(), range(32)):
    ax.imshow(W1[:, k].reshape(IMG_H, IMG_W), cmap="coolwarm"); ax.axis("off")
plt.suptitle("First-layer weights as images (red = positive, blue = negative)"); plt.tight_layout(); plt.show()

## 10. Save and reload

In [ ]:
model2.save("mnist_ann.keras")
loaded = keras.models.load_model("mnist_ann.keras")
same = np.allclose(loaded.predict(X_test_s[:50], verbose=0), model2.predict(X_test_s[:50], verbose=0), atol=1e-5)
print("Reloaded model gives the same predictions:", same)

## Summary

| Step | Key point |
|---|---|
| Data | Images are arrays of pixels (28 x 28 for MNIST), labels are digits 0-9 |
| Preprocessing | Scale to 0-1, flatten to a 784-vector |
| Output layer | 10 nodes + **softmax** (probabilities sum to 1) |
| Loss | **Sparse categorical cross entropy** for integer labels |
| Prediction | `argmax` of the probabilities |
| Evaluation | Accuracy, confusion matrix, per-class precision/recall, look at mistakes |
| Improvements | Dropout, early stopping, more capacity, and a CNN (next step) |

**Limitation of an ANN on images:** flattening throws away the 2D structure (neighbouring pixels), and the number of parameters grows quickly with image size. Convolutional neural networks (CNNs) fix this.

## Practice exercises

1. Change the hidden layers to `(64,)`, `(256, 128)` and `(512, 256, 128)` and compare validation accuracy and parameter counts.
2. Switch to one-hot labels (`keras.utils.to_categorical`) with `categorical_crossentropy` and confirm the results are equivalent.
3. Try `optimizer="sgd"` and compare the speed of learning with Adam.
4. Add a `keras.callbacks.ModelCheckpoint` to keep the best model during training.
5. Shift every test image 2 pixels to the right (`np.roll`) and measure the accuracy drop. Why does an ANN struggle with this?
6. Plot the 10 most confident wrong predictions and describe why they might be confusing.